Generate 5 second images from spectrograms

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import librosa
import librosa.display

# === SET UP ===
trimmed_audio_dir = "/mnt/class_data/Shelby/One_Minute_Audio"
output_dir = "/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/Images_all_classes"
os.makedirs(output_dir, exist_ok=True)

audio_files = [f for f in os.listdir(trimmed_audio_dir) if f.endswith('.WAV') or f.endswith('.wav')]
print(f"Found {len(audio_files)} total files")

csv_path = "/home/Shelby/blackbird_calls/Dataset_processing/Datasets/cv4e_calls_channel1_v2.csv"
annotations_df = pd.read_csv(csv_path)

img_w_px, img_h_px = 1280, 720
dpi = 120
duration_limit = 60  # seconds
segment_length = 5   # seconds

for file_to_visualize in audio_files:
    print(f"Processing: {file_to_visualize}")
    current_wav_fname = file_to_visualize
    matching_annotations = annotations_df[annotations_df["wav_fname"] == current_wav_fname]
    if len(matching_annotations) == 0:
        matching_annotations = annotations_df[annotations_df["wav_fname"].str.lower() == current_wav_fname.lower()]

    audio_path = os.path.join(trimmed_audio_dir, file_to_visualize)
    try:
        y, sr = librosa.load(audio_path, sr=None, mono=False)
        if y.ndim > 1:
            y = y[0]

        total_samples = int(sr * duration_limit)
        y = y[:total_samples]

        num_segments = duration_limit // segment_length
        for i in range(int(num_segments)):
            start_sample = int(i * segment_length * sr)
            end_sample = int((i + 1) * segment_length * sr)
            y_segment = y[start_sample:end_sample]

            fig, ax = plt.subplots(figsize=(img_w_px / dpi, img_h_px / dpi), dpi=dpi)

            n_fft = 1024
            win_length = 1024
            hop_length = 256
            window = "hann"

            D = librosa.stft(y_segment, n_fft=n_fft, hop_length=hop_length, win_length=win_length, window=window, center=False)
            S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

            img = librosa.display.specshow(S_db, sr=sr, hop_length=hop_length, ax=ax, x_axis="off", y_axis="off", shading="nearest", antialiased=False)
            ax.axis('off')

            out_png_fname = f"{Path(file_to_visualize).stem}_spec_{i*segment_length:02d}-{(i+1)*segment_length:02d}.png"
            out_png_path = os.path.join(output_dir, out_png_fname)
            plt.savefig(out_png_path, dpi=dpi, bbox_inches="tight", pad_inches=0)
            plt.close(fig)
            print(f"Saved spectrogram: {out_png_path}")

    except Exception as e:
        print(f"✗ Error processing {file_to_visualize}: {str(e)}")

Found 112 total files
Processing: SL43 Trial 1_trim.WAV
Saved spectrogram: /home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/Images_all_classes/SL43 Trial 1_trim_spec_00-05.png
Saved spectrogram: /home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/Images_all_classes/SL43 Trial 1_trim_spec_05-10.png
Saved spectrogram: /home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/Images_all_classes/SL43 Trial 1_trim_spec_10-15.png
Saved spectrogram: /home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/Images_all_classes/SL43 Trial 1_trim_spec_15-20.png
Saved spectrogram: /home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/Images_all_classes/SL43 Trial 1_trim_spec_20-25.png
Saved spectrogram: /home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/Images_all_classes/SL43 Trial 1_trim_spec_25-30.png
Saved spectrogram: /home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/Images_all_class

: 

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# === PARAMETERS ===
images_dir = "/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/Images_all_classes"
crops_dir = "/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Crops_all_classes"
os.makedirs(crops_dir, exist_ok=True)

csv_path = "/home/Shelby/blackbird_calls/Dataset_processing/Datasets/cv4e_calls_channel1_v2.csv"
annotations_df = pd.read_csv(csv_path)

img_w_px, img_h_px = 1280, 720
segment_length = 5  # seconds
freq_min, freq_max = 0, 24000  # Hz (adjust if needed for your spectrograms)

# Helper: Map time/freq to pixel coordinates
def data_to_pixels(time_s, freq_hz, time_range, freq_range, img_shape):
    x_px = (time_s - time_range[0]) / (time_range[1] - time_range[0]) * img_shape[1]
    y_px = img_shape[0] - (freq_hz - freq_range[0]) / (freq_range[1] - freq_range[0]) * img_shape[0]
    return int(x_px), int(y_px)

# Loop through all images
for img_name in os.listdir(images_dir):
    if not img_name.endswith('.png'):
        continue
    # Parse file info
    stem = Path(img_name).stem
    # Example: SL54 Trial 1_trim_spec_00-05.png
    if '_spec_' not in stem:
        continue
    wav_part, seg_part = stem.split('_spec_')
    # Get segment start/end in seconds
    try:
        seg_start, seg_end = [int(x) for x in seg_part.split('-')]
    except Exception:
        continue
    time_range = (seg_start, seg_end)
    freq_range = (freq_min, freq_max)

    # Find matching annotations for this wav file and segment
    matches = annotations_df[annotations_df['wav_fname'].str.replace('.WAV','').str.replace('.wav','') == wav_part.strip()]
    # Filter to calls within this segment
    matches = matches[(matches['begin_time_s'] < seg_end) & (matches['end_time_s'] > seg_start)]
    if matches.empty:
        continue

    # Load image
    img_path = os.path.join(images_dir, img_name)
    img = Image.open(img_path)
    img_shape = img.size[::-1]  # (height, width)
    
    for idx, row in matches.iterrows():
        # Clip call to segment bounds
        call_start = max(row['begin_time_s'], seg_start)
        call_end = min(row['end_time_s'], seg_end)
        call_low = max(row['low_freq_hz'], freq_min)
        call_high = min(row['high_freq_hz'], freq_max)
        # Map to pixel coordinates
        x0, y1 = data_to_pixels(call_start, call_high, time_range, freq_range, img_shape)
        x1, y0 = data_to_pixels(call_end, call_low, time_range, freq_range, img_shape)
        # Ensure valid crop box
        x0, x1 = sorted([max(0, x0), min(img_shape[1], x1)])
        y0, y1 = sorted([max(0, y0), min(img_shape[0], y1)])
        if x1 - x0 < 5 or y1 - y0 < 5:
            continue  # Skip tiny crops
        crop = img.crop((x0, y0, x1, y1))
        crop_fname = f"{stem}_call_{idx}.png"
        crop.save(os.path.join(crops_dir, crop_fname))
        print(f"Saved crop: {crop_fname}")